In [7]:
import os
import json
import logging
from datetime import datetime

import numpy as np
from numpy.linalg import eigh, svd
import rasterio as rio
from rasterio.windows import Window
from rasterio.enums import Resampling
from rasterio.features import geometry_mask
from rasterio.warp import transform_geom
from rasterio.crs import CRS
from tqdm import tqdm
from joblib import dump
import fiona
from fiona.transform import transform_geom as fiona_transform_geom

from sklearn.decomposition import PCA

In [8]:
# -----------------------------
# CONFIG / PATHS
# -----------------------------
DATA_2019 = r"C:\Users\snvra\NISAR_Data_project\Data\krishsatemb2019_24\krishsatemb2019.tif"
DATA_2024 = r"C:\Users\snvra\NISAR_Data_project\Data\krishsatemb2019_24\krishsatemb2024.tif"
OUTDIR    = r"C:\Users\snvra\NISAR_Data_project\Outputs\Subspace_allignment"
os.makedirs(OUTDIR, exist_ok=True)

STUDY_SHP = r"C:\Users\snvra\NISAR_Data_project\Data\Shapefiles\eastBound\eastBound.shp"
WATER_SHP = r"C:\Users\snvra\NISAR_Data_project\Data\Shapefiles\mask_2024\mask_2024.shp"

TILE = 1024
STABLE_Q = 2      # percentile for stable pixels
RIDGE = 1e-4         # covariance ridge for CORAL
ALIGN_METHOD = "subspace"   # "coral" or "procrustes" or "subspace"

SEED = 42
RNG = np.random.default_rng(SEED)
MAX_Q_SAMPLE = 50000   # maximum number of D_cos values to keep for percentile estimation
MAX_STABLE_PER_WINDOW = 25_000  # max stable pixel samples saved per window
MAX_GLOBAL_STABLE = 50000   # global cap on stable samples to prevent OOM

# Output filenames (in OUTDIR)
STABLE_MASK_PATH = os.path.join(OUTDIR, "stable_mask.tif")
ALIGNED_SCALED_PATH = os.path.join(OUTDIR, "aligned_2019_in2024_scaled.tif")
ALIGNED_RAW_PATH = os.path.join(OUTDIR, "aligned_2019_in2024_raw.tif")
METADATA_JOBLIB = os.path.join(OUTDIR, f"align_metadata_{ALIGN_METHOD}.joblib")
QA_JSON = os.path.join(OUTDIR, "align_qa.json")

# TILE safety
if TILE <= 0:
    TILE = 512

In [9]:
# -----------------------------
# Logging
# -----------------------------
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[logging.StreamHandler()]
)
log = logging.getLogger("domain_align")

# -----------------------------
# Helpers
# -----------------------------
def open_stack(path):
    ds = rio.open(path)
    if ds.count < 1:
        raise AssertionError(f"Expected >=1 band, got {ds.count} for {path}")
    return ds

def same_grid(a, b):
    return (a.crs == b.crs and
            a.transform.almost_equals(b.transform) and
            a.width == b.width and
            a.height == b.height)

def resample_to_match(src_path, ref_ds, out_path, resampling=Resampling.bilinear):
    """Resample src to ref grid and write to out_path."""
    with rio.open(src_path) as src:
        profile = src.profile.copy()
        profile.update({
            "crs": ref_ds.crs,
            "transform": ref_ds.transform,
            "width": ref_ds.width,
            "height": ref_ds.height,
            "dtype": src.dtypes[0]
        })
        with rio.open(out_path, "w", **profile) as dst:
            for b in range(1, src.count + 1):
                rio.warp.reproject(
                    source=rio.band(src, b),
                    destination=rio.band(dst, b),
                    src_transform=src.transform,
                    src_crs=src.crs,
                    dst_transform=ref_ds.transform,
                    dst_crs=ref_ds.crs,
                    resampling=resampling
                )
    return out_path

def windows(ds, tile=TILE):
    """Generator of windows (streaming, no list materialization)."""
    for row in range(0, ds.height, tile):
        h = min(tile, ds.height - row)
        for col in range(0, ds.width, tile):
            w = min(tile, ds.width - col)
            yield Window(col_off=col, row_off=row, width=w, height=h)

# Robust scaling (Median–MAD)
def robust_fit(X, eps=1e-9):
    med = np.median(X, axis=0)
    mad = np.median(np.abs(X - med), axis=0)
    mad = np.where(mad < eps, eps, mad)
    return {"med": med.astype(np.float32), "mad": mad.astype(np.float32)}

def apply_robust(Y, params):
    return (Y - params["med"]) / params["mad"]

def invert_robust(Y_scaled, params):
    return Y_scaled * params["mad"] + params["med"]

# Cosine distance (efficient)
def cosine_distance(a, b, eps=1e-9):
    # a,b: [N, D]
    # handles potential tiny norms
    dot = np.einsum("ij,ij->i", a, b)
    na = np.linalg.norm(a, axis=1)
    nb = np.linalg.norm(b, axis=1)
    denom = (na * nb) + eps
    cos = dot / denom
    return 1.0 - np.clip(cos, -1.0, 1.0)

def cosine_distance_chunks(a, b, chunk=200_000):
    out = np.empty(a.shape[0], dtype=np.float64)
    i = 0
    while i < a.shape[0]:
        j = min(a.shape[0], i + chunk)
        out[i:j] = cosine_distance(a[i:j], b[i:j])
        i = j
    return out

# CORAL helpers (stable PSD sqrt/inv-sqrt)
def _sqrtm_psd(cov, eps=1e-10):
    w, v = eigh(cov + eps * np.eye(cov.shape[0]))
    w = np.clip(w, eps, None)
    return (v * np.sqrt(w)) @ v.T

def _invsqrtm_psd(cov, eps=1e-10):
    w, v = eigh(cov + eps * np.eye(cov.shape[0]))
    w = np.clip(w, eps, None)
    return (v * (1.0 / np.sqrt(w))) @ v.T

def coral_fit(X_src, X_tgt, ridge=RIDGE, ddof=0):
    mu_s = X_src.mean(0)
    mu_t = X_tgt.mean(0)
    Cs = np.cov(X_src, rowvar=False, ddof=ddof) + ridge * np.eye(X_src.shape[1])
    Ct = np.cov(X_tgt, rowvar=False, ddof=ddof) + ridge * np.eye(X_tgt.shape[1])
    As = _invsqrtm_psd(Cs)
    Bt = _sqrtm_psd(Ct)
    return {"mu_s": mu_s.astype(np.float32),
            "mu_t": mu_t.astype(np.float32),
            "As": As.astype(np.float32),
            "Bt": Bt.astype(np.float32)}

def coral_apply(X, params):
    return (X - params["mu_s"]) @ params["As"] @ params["Bt"] + params["mu_t"]

#Sub_Space Allignment helpers

def sa_fit(X_src, X_tgt, n_components=10):
    """
    Subspace Alignment (Fernando et al.)

    X_src : stable source samples
    X_tgt : stable target samples
    """

    pca_s = PCA(n_components=n_components, random_state=42)
    pca_t = PCA(n_components=n_components, random_state=42)

    Xs_sub = pca_s.fit(X_src).components_.T.astype(np.float32)
    Xt_sub = pca_t.fit(X_tgt).components_.T.astype(np.float32)

    M = (Xs_sub.T @ Xt_sub).astype(np.float32)

    XA = (Xs_sub @ M).astype(np.float32)

    return {
        "d": n_components,
        "Xs_sub": Xs_sub,
        "Xt_sub": Xt_sub,
        "M": M,
        "XA": XA
    }
def sa_apply(X, params):

    XA = params["XA"]
    Xt_sub = params["Xt_sub"]

    # Project source into aligned subspace
    Z = X @ XA

    # Reconstruct back into 64-D target space
    X_rec = Z @ Xt_sub.T

    return X_rec.astype(np.float32)
# Procrustes
def procrustes_fit(X_src, X_tgt):
    mu_s = X_src.mean(0); mu_t = X_tgt.mean(0)
    Xs = X_src - mu_s
    Xt = X_tgt - mu_t
    U, _, Vt = svd(Xs.T @ Xt, full_matrices=False)
    R = U @ Vt
    # avoid reflection if we want pure rotation
    if np.linalg.det(R) < 0:
        U[:, -1] *= -1
        R = U @ Vt
    return {"mu_s": mu_s.astype(np.float32), "mu_t": mu_t.astype(np.float32), "R": R.astype(np.float32)}

def procrustes_apply(X, params):
    return (X - params["mu_s"]) @ params["R"] + params["mu_t"]

# Geometry loader + reprojection
def load_and_reproject_geoms(shp_path, dst_crs):
    """Load geometries from shapefile and reproject them to dst_crs (rasterio CRS or string)."""
    geoms = []
    dst_crs_obj = CRS.from_user_input(dst_crs)
    if not os.path.exists(shp_path):
        raise FileNotFoundError(f"Shapefile not found: {shp_path}")
    with fiona.open(shp_path, "r") as src:
        src_crs = src.crs if src.crs else src.crs_wkt
        for feat in src:
            geom = feat["geometry"]
            if src_crs:
                try:
                    g2 = transform_geom(src_crs, dst_crs_obj.to_string(), geom, precision=6)
                except Exception:
                    # last resort, use original geometry (assume already in dst_crs)
                    g2 = geom
            else:
                g2 = geom
            geoms.append(g2)
    return geoms

# Reservoir sampling support for percentile estimation
def reservoir_add(reservoir_list, values, max_size, rng, total_seen):
    """Append values into reservoir_list using reservoir sampling; returns updated total_seen."""
    # reservoir_list: python list
    # values: 1D numpy array
    for v in values:
        total_seen += 1
        if len(reservoir_list) < max_size:
            reservoir_list.append(float(v))
        else:
            j = rng.integers(0, total_seen)
            if j < max_size:
                reservoir_list[j] = float(v)
    return total_seen

In [10]:
# -----------------------------
# OPEN & ALIGN GRIDS
# -----------------------------
log.info("Starting domain alignment script")

ds_ref = open_stack(DATA_2024)
ds_toMatch = open_stack(DATA_2019)

if not same_grid(ds_ref, ds_toMatch):
    log.info("Grids differ — resampling 2019 to 2024 grid...")
    data_2019_RES = os.path.join(OUTDIR, "emb_2019_resampled.tif")
    resample_to_match(DATA_2019, ds_ref, data_2019_RES, resampling=Resampling.bilinear)
    ds_toMatch.close()
    ds_toMatch = rio.open(data_2019_RES)

# -----------------------------
# Create study & water masks
# -----------------------------
log.info("Creating study-area and water masks from shapefiles...")
study_geoms = load_and_reproject_geoms(STUDY_SHP, ds_ref.crs)
water_geoms = load_and_reproject_geoms(WATER_SHP, ds_ref.crs)

inside_study = geometry_mask(study_geoms,
                             transform=ds_ref.transform,
                             invert=True,
                             out_shape=(ds_ref.height, ds_ref.width))

inside_water = geometry_mask(water_geoms,
                             transform=ds_ref.transform,
                             invert=True,
                             out_shape=(ds_ref.height, ds_ref.width))

processing_mask = inside_study & (~inside_water)
del inside_study, inside_water
log.info("Masks created")

# -----------------------------
# STABLE PIXELS BY D_cos (reservoir sampling)
# -----------------------------
log.info("Estimating D_cos threshold with reservoir sampling...")
q_reservoir = []
total_d_seen = 0

# iterate streaming windows (no list(...))
for w in tqdm(windows(ds_ref, TILE), desc="D_cos percentile sample"):
    A = ds_ref.read(window=w).astype(np.float32)
    B = ds_toMatch.read(window=w).astype(np.float32)
    A = np.moveaxis(A, 0, -1)
    B = np.moveaxis(B, 0, -1)

    rs, cs = w.toslices()
    mask_chunk = processing_mask[rs, cs]

    valid_geom = mask_chunk
    valid_data = np.isfinite(A).all(axis=-1) & np.isfinite(B).all(axis=-1)
    valid = valid_geom & valid_data

    if not valid.any():
        continue

    flatA = A.reshape(-1, A.shape[-1])
    flatB = B.reshape(-1, B.shape[-1])
    valid_idx = np.where(valid.ravel())[0]
    Asel = flatA[valid_idx]
    Bsel = flatB[valid_idx]

    # compute d in chunks if large
    if Asel.shape[0] > 300_000:
        d = cosine_distance_chunks(Asel, Bsel, chunk=200_000)
    else:
        d = cosine_distance(Asel, Bsel)

    # sample from d into reservoir (cap memory)
    if d.size > 50_000:
        sel_idx = RNG.choice(d.size, 50_000, replace=False)
        vals = d[sel_idx]
    else:
        vals = d
    total_d_seen = reservoir_add(q_reservoir, vals, MAX_Q_SAMPLE, RNG, total_d_seen)

if len(q_reservoir) == 0:
    raise RuntimeError("No valid pixels found for D_cos sampling. Check shapefiles and alignment.")

q_arr = np.array(q_reservoir, dtype=np.float64)
thr = float(np.percentile(q_arr, STABLE_Q))
log.info(f"D_cos threshold (percentile={STABLE_Q:.3f}): {thr:.6f}")

# -----------------------------
# Write stable mask and collect stable samples (streaming; with caps)
# -----------------------------
log.info("Collecting stable samples and writing stable mask...")
profile = ds_ref.profile.copy()
profile.update(count=1, dtype="uint8", nodata=0)

X24_stable_list = []
X19_stable_list = []
global_stable_count = 0

with rio.open(STABLE_MASK_PATH, "w", **profile) as dst_mask:
    for w in tqdm(windows(ds_ref, TILE), desc="Stable mask + samples"):
        A = ds_ref.read(window=w).astype(np.float32)
        B = ds_toMatch.read(window=w).astype(np.float32)
        A = np.moveaxis(A, 0, -1)
        B = np.moveaxis(B, 0, -1)

        rs, cs = w.toslices()
        mask_chunk = processing_mask[rs, cs]

        valid = mask_chunk & np.isfinite(A).all(axis=-1) & np.isfinite(B).all(axis=-1)
        flatA = A.reshape(-1, A.shape[-1])
        flatB = B.reshape(-1, B.shape[-1])

        vidx = np.where(valid.ravel())[0]
        if vidx.size == 0:
            dst_mask.write(np.zeros((A.shape[0], A.shape[1]), dtype=np.uint8), window=w, indexes=1)
            continue

        # compute distances only on valid rows
        if vidx.size > 300_000:
            # compute in chunks
            d = cosine_distance_chunks(flatA[vidx], flatB[vidx], chunk=200_000)
        else:
            d = cosine_distance(flatA[vidx], flatB[vidx])

        # produce full-window d array initialized to large
        d_full = np.full(flatA.shape[0], np.inf, dtype=d.dtype)
        d_full[vidx] = d
        d2 = d_full.reshape(A.shape[0], A.shape[1])

        # stable mask for this window
        m = ((d2 <= thr) & valid).astype(np.uint8)
        dst_mask.write(m, window=w, indexes=1)

        idx = np.where(m.ravel() == 1)[0]
        if idx.size > 0 and global_stable_count < MAX_GLOBAL_STABLE:
            take = min(MAX_STABLE_PER_WINDOW, idx.size)
            sel = RNG.choice(idx, take, replace=False)
            X24_stable_list.append(flatA[sel])
            X19_stable_list.append(flatB[sel])
            global_stable_count += take

log.info(f"Collected approx {global_stable_count:,} stable samples (capped to {MAX_GLOBAL_STABLE})")

if len(X24_stable_list) == 0 or len(X19_stable_list) == 0:
    raise RuntimeError("No stable samples collected after masking. Lower STABLE_Q or inspect masks/shapefiles.")

X24_stable = np.concatenate(X24_stable_list, axis=0)
X19_stable = np.concatenate(X19_stable_list, axis=0)
log.info(f"Stable samples shape: X24 {X24_stable.shape}, X19 {X19_stable.shape}")

# -----------------------------
# 2) FIT ROBUST SCALERS (year-wise on stable pixels)
# -----------------------------
log.info("Fitting robust (median-MAD) scalers on stable samples...")
sc24_params = robust_fit(X24_stable)
sc19_params = robust_fit(X19_stable)
dump({"sc24": sc24_params, "sc19": sc19_params}, os.path.join(OUTDIR, "scaler_params.joblib"))
log.info("Saved scaler parameters")

X24s = apply_robust(X24_stable, sc24_params)
X19s = apply_robust(X19_stable, sc19_params)

# -----------------------------
# 3) FIT ALIGNMENT (CORAL or Procrustes) ON SCALED STABLE SAMPLES
# -----------------------------
log.info(f"Fitting alignment method: {ALIGN_METHOD}")
if ALIGN_METHOD.lower() == "coral":
    align_params = coral_fit(X19s, X24s, ridge=RIDGE)
    align_kind = "coral"
elif ALIGN_METHOD.lower() == "subspace":
    align_params = sa_fit(
        X19s,
        X24s,
        n_components=10
    )
    align_kind = "subspace"    
elif ALIGN_METHOD.lower() == "procrustes":
    align_params = procrustes_fit(X19s, X24s)
    align_kind = "procrustes"
else:
    raise ValueError("ALIGN_METHOD must be 'coral' or 'procrustes' or 'subspace' ")

dump({"method": align_kind, "params": align_params}, METADATA_JOBLIB)
log.info(f"Saved alignment metadata to {METADATA_JOBLIB}")
# -----------------------------
# 4) APPLY ALIGNMENT TO FULL 2019 RASTER (WRITE TWO PRODUCTS)
#    A) aligned_2019_in_2024_scaled.tif (in 2024 scaled space)
#    B) aligned_2019_in_2024_raw.tif    (mapped back to 2024 raw space)
# -----------------------------
log.info("Applying alignment to full 2019 raster and writing outputs...")
profile64 = ds_ref.profile.copy()
profile64.update(count=ds_ref.count, dtype="float32", nodata=None, compress="deflate")

with rio.open(ALIGNED_SCALED_PATH, "w", **profile64) as dst_scaled, \
     rio.open(ALIGNED_RAW_PATH,    "w", **profile64) as dst_raw:

    for w in tqdm(windows(ds_toMatch, TILE), desc="Apply alignment to 2019"):
        arr19 = ds_toMatch.read(window=w).astype(np.float32)     # [bands,h,w]
        arr19_hw = np.moveaxis(arr19, 0, -1)                     # [h,w,bands]
        flat19   = arr19_hw.reshape(-1, arr19_hw.shape[-1])

        rs, cs = w.toslices()
        mask_chunk = processing_mask[rs, cs]
        if not mask_chunk.any():
            # Outside study: write raw product and also produce a scaled block (identity alignment)
            dst_raw.write(arr19, window=w)

            # Produce scaled unaligned (apply robust scaling only)
            flat19_s = apply_robust(flat19, sc19_params)
            block_scaled = flat19_s.reshape(arr19_hw.shape[0], arr19_hw.shape[1], arr19_hw.shape[-1])
            block_scaled = np.moveaxis(block_scaled, -1, 0).astype(np.float32)
            dst_scaled.write(block_scaled, window=w)
            continue

        # For inside-study windows: proceed normally
        flat19_s = apply_robust(flat19, sc19_params)

        # Align in scaled space
        if align_kind == "coral":
            flat_aligned_s = coral_apply(flat19_s, align_params)
        elif align_kind == "subspace":
            flat_aligned_s = sa_apply(flat19_s, align_params)
        else:
            flat_aligned_s = procrustes_apply(flat19_s, align_params)

        # Write scaled-space aligned product
        block_scaled = flat_aligned_s.reshape(arr19_hw.shape[0], arr19_hw.shape[1], arr19_hw.shape[-1])
        block_scaled = np.moveaxis(block_scaled, -1, 0).astype(np.float32)  # [bands,h,w]
        dst_scaled.write(block_scaled, window=w)

        # Map back to 2024 RAW (inverse robust using sc24)
        flat_aligned_raw = invert_robust(flat_aligned_s, sc24_params)
        block_raw = flat_aligned_raw.reshape(arr19_hw.shape[0], arr19_hw.shape[1], arr19_hw.shape[-1])
        block_raw = np.moveaxis(block_raw, -1, 0).astype(np.float32)
        dst_raw.write(block_raw, window=w)

log.info(f"Wrote scaled aligned product: {ALIGNED_SCALED_PATH}")
log.info(f"Wrote raw mapped product:     {ALIGNED_RAW_PATH}")
log.info(f"Wrote stable mask:           {STABLE_MASK_PATH}")

# -----------------------------
# 5) QUICK QA: covariance gap before/after, per-band stats
# -----------------------------
def cov_gap(A, B):
    CA = np.cov(A, rowvar=False)
    CB = np.cov(B, rowvar=False)
    return float(np.linalg.norm(CA - CB, ord='fro'))

log.info("Computing QA statistics on stable samples...")
gap_before = cov_gap(X19s, X24s)
if align_kind == "coral":
    X19s_al = coral_apply(X19s, align_params)
elif align_kind == "subspace":
    X19s_al = sa_apply(X19s, align_params)    
else:
    X19s_al = procrustes_apply(X19s, align_params)
gap_after = cov_gap(X19s_al, X24s)

# per-band stats on raw stable samples
per_band_stats = []
for i in range(X24_stable.shape[1]):
    s24 = X24_stable[:, i]
    s19 = X19_stable[:, i]
    per_band_stats.append({
        "band": int(i + 1),
        "X24_median": float(np.median(s24)),
        "X24_mad": float(np.median(np.abs(s24 - np.median(s24)))),
        "X19_median": float(np.median(s19)),
        "X19_mad": float(np.median(np.abs(s19 - np.median(s19))))
    })

sa_before = None
sa_after = None

if align_kind == "subspace":
    sa_before = float(
        np.linalg.norm(
            align_params["Xs_sub"] -
            align_params["Xt_sub"]
        )
    )

    sa_after = float(
        np.linalg.norm(
            align_params["XA"] -
            align_params["Xt_sub"]
        )
    )    
    
    qa = {
    "date_utc": datetime.utcnow().isoformat() + "Z",
    "align_method": align_kind,
    "percentile for stable pixels": STABLE_Q,
    "D_cos_threshold": thr,
    "stable_sample_count": int(X24_stable.shape[0]),
    "seed": int(SEED),
    "cov_gap_before_scaled": gap_before,
    "cov_gap_after_scaled": gap_after,
    "sa_subspace_before": sa_before,
    "sa_subspace_after": sa_after,
    "per_band_stats": per_band_stats
}

with open(QA_JSON, "w") as fh:
    json.dump(qa, fh, indent=2)

log.info(f"QA saved to {QA_JSON}")
log.info("Domain alignment complete. Proceed to classification when ready.")

2026-06-11 11:33:46,973 [INFO] Starting domain alignment script
2026-06-11 11:33:47,104 [WARNING] CPLE_AppDefined in krishsatemb2024.tif: TIFFReadDirectory:Sum of Photometric type-related color channels and ExtraSamples doesn't match SamplesPerPixel. Defining non-color channels as ExtraSamples.
2026-06-11 11:33:47,220 [WARNING] CPLE_AppDefined in krishsatemb2019.tif: TIFFReadDirectory:Sum of Photometric type-related color channels and ExtraSamples doesn't match SamplesPerPixel. Defining non-color channels as ExtraSamples.
2026-06-11 11:33:47,222 [INFO] Creating study-area and water masks from shapefiles...
2026-06-11 11:33:55,627 [INFO] Masks created
2026-06-11 11:33:55,628 [INFO] Estimating D_cos threshold with reservoir sampling...
D_cos percentile sample: 9it [00:25,  2.85s/it]
2026-06-11 11:34:21,373 [INFO] D_cos threshold (percentile=2.000): 0.014274
2026-06-11 11:34:21,384 [INFO] Collecting stable samples and writing stable mask...
2026-06-11 11:34:21,567 [WARNING] CPLE_IllegalAr

In [11]:
print("X19s:", X19s.shape)
print("X24s:", X24s.shape)

X19s: (41937, 64)
X24s: (41937, 64)


In [12]:
from sklearn.decomposition import PCA

d = 20   # initial subspace dimension

pca_s = PCA(n_components=d, random_state=42)
pca_t = PCA(n_components=d, random_state=42)

Xs_sub = pca_s.fit(X19s).components_.T
Xt_sub = pca_t.fit(X24s).components_.T

print("Xs_sub:", Xs_sub.shape)
print("Xt_sub:", Xt_sub.shape)

Xs_sub: (64, 20)
Xt_sub: (64, 20)


In [13]:
M = Xs_sub.T @ Xt_sub

print("M:", M.shape)

M: (20, 20)


In [14]:
XA = Xs_sub @ M

print("XA:", XA.shape)

XA: (64, 20)


In [15]:
import numpy as np

before = np.linalg.norm(Xs_sub - Xt_sub)
after = np.linalg.norm(XA - Xt_sub)

print("Subspace distance BEFORE:", before)
print("Subspace distance AFTER :", after)

Subspace distance BEFORE: 6.1836696
Subspace distance AFTER : 1.2677847


In [16]:
# Source projected into aligned space
X19_sa = X19s @ XA

# Target projected into target subspace
X24_sa = X24s @ Xt_sub

In [17]:
import numpy as np

dist_after = np.linalg.norm(
    np.mean(X19_sa, axis=0) -
    np.mean(X24_sa, axis=0)
)

print(dist_after)

0.5534999


In [18]:
def cov_gap(A, B):
    CA = np.cov(A, rowvar=False)
    CB = np.cov(B, rowvar=False)
    return float(np.linalg.norm(CA - CB, ord='fro'))

In [19]:
import numpy as np
import pandas as pd

results = []

# constant before-covariance
cov_before = cov_gap(X19s, X24s)

for d in range(1, 65):

    params = sa_fit(X19s, X24s, n_components=d)

    # Subspace distances
    sa_before = np.linalg.norm(
        params["Xs_sub"] - params["Xt_sub"]
    )

    sa_after = np.linalg.norm(
        params["XA"] - params["Xt_sub"]
    )

    # Apply SA
    X19s_al = sa_apply(X19s, params)

    # Covariance distances
    cov_after = cov_gap(X19s_al, X24s)

    results.append({
        "d": d,

        "cov_before": float(cov_before),
        "cov_after": float(cov_after),
        "cov_improvement": float(cov_before - cov_after),

        "sa_before": float(sa_before),
        "sa_after": float(sa_after),
        "sa_improvement": float(sa_before - sa_after)
    })

    print(
        f"d={d:2d} | "
        f"Cov: {cov_before:.3f}->{cov_after:.3f} | "
        f"SA: {sa_before:.3f}->{sa_after:.3f}"
    )

df = pd.DataFrame(results)

df.to_csv("SA_dimension_sweep.csv", index=False)

print(df.head())

d= 1 | Cov: 18.443->29.196 | SA: 0.121->0.121
d= 2 | Cov: 18.443->25.469 | SA: 0.269->0.253
d= 3 | Cov: 18.443->23.084 | SA: 1.848->0.773
d= 4 | Cov: 18.443->18.297 | SA: 2.608->0.446
d= 5 | Cov: 18.443->15.295 | SA: 3.267->0.449
d= 6 | Cov: 18.443->13.918 | SA: 3.817->0.479
d= 7 | Cov: 18.443->12.546 | SA: 3.826->0.493
d= 8 | Cov: 18.443->12.100 | SA: 3.839->0.545
d= 9 | Cov: 18.443->12.090 | SA: 4.199->1.020
d=10 | Cov: 18.443->12.058 | SA: 4.520->0.707
d=11 | Cov: 18.443->12.342 | SA: 4.913->0.795
d=12 | Cov: 18.443->12.389 | SA: 5.006->1.102
d=13 | Cov: 18.443->13.152 | SA: 5.305->0.938
d=14 | Cov: 18.443->13.527 | SA: 5.496->1.262
d=15 | Cov: 18.443->13.589 | SA: 5.658->1.373
d=16 | Cov: 18.443->13.674 | SA: 5.792->1.309
d=17 | Cov: 18.443->13.650 | SA: 5.900->1.248
d=18 | Cov: 18.443->13.827 | SA: 6.026->1.318
d=19 | Cov: 18.443->14.097 | SA: 6.112->1.248
d=20 | Cov: 18.443->14.177 | SA: 6.184->1.268
d=21 | Cov: 18.443->14.223 | SA: 6.421->1.219
d=22 | Cov: 18.443->15.170 | SA: 6

In [20]:
coral_params = coral_fit(X19s, X24s)
X19_coral = coral_apply(X19s, coral_params)

print("CORAL covariance gap:", cov_gap(X19_coral, X24s))


CORAL covariance gap: 0.0017522239709131598


In [21]:
def sa_metrics(params):
    before = np.linalg.norm(params["Xs_sub"] - params["Xt_sub"])
    after = np.linalg.norm(params["XA"] - params["Xt_sub"])
    return before, after

In [22]:
params = sa_fit(X19_coral, X24s, n_components=10)

before, after = sa_metrics(params)

print("SA distance before:", before)
print("SA distance after :", after)

SA distance before: 0.00027175373
SA distance after : 0.00016866755


In [23]:
sa_params = sa_fit(
    X19s,
    X24s,
    n_components=8
)

X19_sa = sa_apply(
    X19s,
    sa_params
)

In [25]:
coral_params2 = coral_fit(
    X19_sa,
    X24s
)

X19_sa_coral = coral_apply(
    X19_sa,
    coral_params2
)

In [26]:
print(
    "Covariance:",
    cov_gap(X19_sa_coral, X24s)
)

Covariance: 5.414516345951118


In [27]:
print(
    "SA distance:",
    np.linalg.norm(
        sa_params["XA"] -
        sa_params["Xt_sub"]
    )
)

SA distance: 0.5449392


In [28]:
cov_original = cov_gap(X19s, X24s)

print("Original:", cov_original)

Original: 18.443347393903537


In [29]:
coral_params = coral_fit(X19s, X24s)

X19_coral = coral_apply(X19s, coral_params)

cov_coral = cov_gap(
    X19_coral,
    X24s
)

print("After CORAL:", cov_coral)

After CORAL: 0.0017522239709131598


In [30]:
sa_params = sa_fit(
    X19_coral,
    X24s,
    n_components=10
)

X19_coral_sa = sa_apply(
    X19_coral,
    sa_params
)

In [31]:
cov_coral_sa = cov_gap(
    X19_coral_sa,
    X24s
)

print("After CORAL+SA:", cov_coral_sa)

After CORAL+SA: 3.9024537007368947


In [32]:
from adapt.feature_based import TCA

In [33]:
import numpy as np

n = 5000
rng = np.random.default_rng(42)

idx_s = rng.choice(len(X19s), n, replace=False)
idx_t = rng.choice(len(X24s), n, replace=False)

Xs = X19s[idx_s]
Xt = X24s[idx_t]

print(Xs.shape, Xt.shape)

(5000, 64) (5000, 64)


In [35]:
from adapt.feature_based import TCA

tca = TCA()
print(tca)    

TCA()


In [37]:
from adapt.feature_based import TCA

help(TCA)

Help on class TCA in module adapt.feature_based._tca:

class TCA(adapt.base.BaseAdaptEstimator)
 |  TCA(
 |      estimator=None,
 |      Xt=None,
 |      n_components=20,
 |      mu=0.1,
 |      kernel='rbf',
 |      copy=True,
 |      verbose=1,
 |      random_state=None,
 |      **params
 |  )
 |
 |  TCA :  Transfer Component Analysis
 |
 |  Parameters
 |  ----------
 |      estimator : sklearn estimator or tensorflow Model (default=None)
 |          Estimator used to learn the task.
 |          If estimator is ``None``, a ``LinearRegression``
 |          instance is used as estimator.
 |
 |      Xt : numpy array (default=None)
 |          Target input data.
 |
 |  n_components : int or float (default=None)
 |      Number of components to keep.
 |
 |  mu : float (default=0.1)
 |      Regularization parameter. The larger
 |      ``mu`` is, the less adaptation is performed.
 |
 |      copy : boolean (default=True)
 |          Whether to make a copy of ``estimator`` or not.
 |
 |      v

In [38]:
from adapt.feature_based import TCA

tca = TCA(
    n_components=10
)

Xtca = tca.fit_transform(Xs, Xt)

In [39]:
from adapt.feature_based import TCA

tca = TCA(
    Xt=Xt,
    n_components=10,
    mu=0.1,
    kernel="rbf",
    verbose=0
)

Xs_tca = tca.fit_transform(Xs, Xt)
Xt_tca = tca.transform(Xt, domain="tgt")

print(Xs_tca.shape)
print(Xt_tca.shape)

(5000, 10)
(5000, 10)


In [40]:
cov_before = cov_gap(Xs, Xt)
cov_after  = cov_gap(Xs_tca, Xt_tca)

print("Cov before:", cov_before)
print("Cov after :", cov_after)

Cov before: 18.525055194227193
Cov after : 6.182421050776759


In [41]:
from sklearn.decomposition import PCA
import numpy as np

d = 10

Ps = PCA(n_components=d).fit(Xs_tca).components_.T
Pt = PCA(n_components=d).fit(Xt_tca).components_.T

subspace_dist = np.linalg.norm(Ps - Pt)

print("TCA subspace distance:", subspace_dist)

TCA subspace distance: 0.8686638272139445
